# SAMPIC timing and readout analysis

`ST` event timing and `SC` collector timing are optional. `SR` describes periodic host readout ticks; `SB` describes fixed bins in hardware time. These clock domains are never subtracted from one another.

The data-volume estimate reconstructs **uncompressed MIDAS bytes** for decoded banks, including headers and alignment. ROOT `GetEntry()` bytes are not used as a DAQ data-rate measurement. Rates derived from frontend timestamp spacing are production-cadence estimates, not hardware self-trigger rates or measured disk/network throughput.


In [ ]:
# Edit this path, then run the notebook from the top.
# Absolute paths and ~ are supported; relative paths start at the app directory.
DATA_FILE = ".artifacts/bank_format_verification/run00266/output.root"


In [ ]:
from pathlib import Path
import sys, os
# Works when Jupyter starts in the repository or a notebook subdirectory.
APP_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
                if (p / 'notebooks/notebook_support.py').is_file())
if str(APP_ROOT / 'notebooks') not in sys.path:
    sys.path.insert(0, str(APP_ROOT / 'notebooks'))
from notebook_support import (open_data, present, hit_waveform, hit_baseline,
    hit_amplitude, hit_peak, waveform_entries as find_waveform_entries,
    sampic_unit, integer, waveform_packets, sampic_event_bytes)
import ROOT
r = ROOT
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict, Counter
root_file, tree, DATA_FILE = open_data('SAMPIC', DATA_FILE)
f = file = root_file
plt.rcParams.update({'figure.figsize': (12, 5), 'axes.grid': True, 'grid.alpha': 0.25})
UNIT = sampic_unit(tree)
print('Waveform display units:', UNIT)


In [ ]:
timing, collector, ticks, bins, hit_times = [], [], [], [], []
wire_bytes = []
for idx in range(tree.GetEntries()):
    tree.GetEntry(idx)
    wire_bytes.append(sampic_event_bytes(tree))
    if present(tree, 'event'):
        hit_times.extend((idx, float(h.first_cell_timestamp)) for h in tree.sampic_event.hits)
    if present(tree, 'event_timing'):
        t = tree.sampic_event_timing
        row = {k:int(getattr(t,k)) for k in ('fe_timestamp_ns','nhits','nparents','sp_prepare_us_max',
            'sp_read_us_max','sp_decode_us_max','sp_total_us_max','sp_acq_retry_max','sp_acq_retry_sum')}
        row.update(entry=idx, bytes=wire_bytes[-1]); timing.append(row)
    if present(tree, 'collector_timing'):
        t = tree.sampic_collector_timing
        collector.append({k:int(getattr(t,k)) for k in ('wait_us','group_build_us','finalize_us','total_us','n_events','total_hits')})
    if present(tree, 'readout_tick'):
        t = tree.sampic_readout_tick
        ticks.append({k:int(getattr(t,k)) for k in ('first_tick','ticks_elapsed','start_ns','end_ns','actual_readout_ns','flags','hit_count','parent_count')})
    if present(tree, 'time_bin'):
        t = tree.sampic_time_bin
        bins.append(dict(index=int(t.bin_index), start=float(t.start_ns), end=float(t.end_ns), hits=int(t.hit_count)))
print(f'Entries: {tree.GetEntries():,}; ST: {len(timing):,}; SC: {len(collector):,}; SR: {len(ticks):,}; SB: {len(bins):,}')
print(f'Known-bank MIDAS volume: {sum(wire_bytes)/1e6:.3f} MB (no compression, ODB records, or unknown banks)')


In [ ]:
if timing:
    fig, axes = plt.subplots(2,3,figsize=(15,8))
    names = ['sp_prepare_us_max','sp_read_us_max','sp_decode_us_max','sp_total_us_max','sp_acq_retry_max','sp_acq_retry_sum']
    for ax,name in zip(axes.flat,names):
        values = np.array([x[name] for x in timing])
        ax.hist(values,bins=40); ax.set(xlabel=name, ylabel='Events')
        print(f'{name}: mean={values.mean():.3f}, min={values.min()}, max={values.max()}')
    plt.tight_layout(); plt.show()
    fig, axes = plt.subplots(1,2,figsize=(12,4))
    for ax,name in zip(axes,('nhits','nparents')):
        ax.scatter([x[name] for x in timing],[x['sp_total_us_max'] for x in timing],s=5)
        ax.set(xlabel=name, ylabel='Max parent processing time [us]')
    plt.tight_layout(); plt.show()
else: print('ST event timing is absent.')
if collector:
    fig, axes = plt.subplots(2,2,figsize=(12,7))
    for ax,name in zip(axes.flat,('wait_us','group_build_us','finalize_us','total_us')):
        ax.hist([x[name] for x in collector],bins=40); ax.set(xlabel=name,ylabel='Collection cycles')
    plt.tight_layout(); plt.show()
else: print('SC collector timing is absent (normal for periodic mode).')


In [ ]:
if len(timing) > 1:
    times = np.array([x['fe_timestamp_ns'] for x in timing],dtype=np.int64)
    entries = np.array([x['entry'] for x in timing])
    dt = np.diff(times)*1e-9
    # Only adjacent tree rows have a complete byte/hit interval for this estimate.
    valid = (dt > 0) & (np.diff(entries) == 1)
    rates = 1/dt[valid]
    byte_rates = np.array([x['bytes'] for x in timing[1:]])[valid]/dt[valid]/1e6
    fig, axes = plt.subplots(2,2,figsize=(12,7))
    axes[0,0].plot(entries[1:][valid],rates,'.'); axes[0,0].set(xlabel='Tree entry',ylabel='1 / event spacing [Hz]')
    axes[0,1].hist(rates,bins=40); axes[0,1].set(xlabel='1 / event spacing [Hz]',ylabel='Intervals')
    axes[1,0].plot(entries[1:][valid],byte_rates,'.'); axes[1,0].set(xlabel='Tree entry',ylabel='Known-bank cadence estimate [MB/s]')
    axes[1,1].hist(byte_rates,bins=40); axes[1,1].set(xlabel='Known-bank cadence estimate [MB/s]',ylabel='Intervals')
    plt.tight_layout(); plt.show()
    print('Excluded nonpositive/nonadjacent intervals:', np.count_nonzero(~valid))
    if np.all(valid):
        duration = (times[-1]-times[0])*1e-9
        print(f'Cadence averages: {(len(times)-1)/duration:.3f} events/s; '
              f'{sum(x["nhits"] for x in timing[1:])/duration:.3f} hits/s; '
              f'{sum(x["bytes"] for x in timing[1:])/duration/1e6:.3f} MB/s')
else: print('Not enough ST timestamps for cadence estimates.')
if hit_times:
    values = np.asarray(hit_times)
    fig, ax = plt.subplots(); ax.plot(values[:,0],values[:,1],'.',markersize=1)
    ax.set(xlabel='Tree entry',ylabel='Hardware first-cell timestamp [ns]'); plt.show()


In [ ]:
if ticks:
    starts = np.array([x['start_ns'] for x in ticks],dtype=np.int64)
    ends = np.array([x['end_ns'] for x in ticks],dtype=np.int64)
    actual = np.array([x['actual_readout_ns'] for x in ticks],dtype=np.int64)
    print('SR late/coalesced/final events:', [sum(bool(x['flags'] & bit) for x in ticks) for bit in (1,2,4)])
    fig, axes = plt.subplots(1,3,figsize=(15,4))
    axes[0].plot((ends-starts)*1e-6,'.'); axes[0].set(xlabel='Readout event',ylabel='Scheduled interval [ms]')
    axes[1].plot((actual-ends)*1e-3,'.'); axes[1].set(xlabel='Readout event',ylabel='Readout minus scheduled end [us]')
    axes[2].plot([x['hit_count'] for x in ticks],'.'); axes[2].set(xlabel='Readout event',ylabel='Hits')
    plt.tight_layout(); plt.show()
else: print('SR periodic-readout metadata is absent.')
if bins:
    fig, ax = plt.subplots(); ax.plot([x['index'] for x in bins],[x['hits'] for x in bins],'.')
    ax.set(xlabel='Hardware bin index',ylabel='Hits'); plt.show()
    print('Hardware bin widths [ns]:', sorted(set(x['end']-x['start'] for x in bins)))
else: print('SB fixed-time-bin metadata is absent.')
